# 05 PPO：标准 Actor-Critic、GAE 与 PPO-Clip

本节自行实现标准在线 PPO 教学基线。Actor 生成回答，Critic 估计每个 Response Token 的状态价值，Old Policy 保存行为策略 Log Probability，确定性 Verifier 提供 Outcome Reward。PPO 与 GRPO 分开：PPO 使用 Critic 和 GAE，不依赖组内相对奖励。

DGX Spark 阶段只运行 CPU 数值 Smoke Test；不加载 Qwen3、不启动 vLLM、不进行 GPU Rollout 或更新。真实实验在已核实的 NVIDIA GPU 上使用独立训练/vLLM Profile 同步串行执行；不能将 vGPU 性能等同于 RTX 5090。


当前默认基座为 `Qwen/Qwen3-0.6B`，保留官方 Thinking 模板。原 GPU Baseline 与 SFT/DPO/PPO/GRPO 资源 Smoke 已完成；新版低显存实现已通过本地 CPU 检查，尚待 GPU 复测。见 [基准报告](docs/QWEN3_0_6B_BENCHMARK.md) 与 [低显存说明](docs/QWEN3_0_6B_LOW_MEMORY.md)。

第二轮正式设计见 [实验协议](docs/ROUND2_EXPERIMENT_PROTOCOL.md)：当前完成诊断与CPU准备，GPU未运行，等待人工审核。旧 standalone Smoke 与正式 Runner 的调度不同；正式配置来自 `configs/round2_design.json`。



## 学习目标

- 对齐 Prompt、Completion、EOS、Padding 和 Response Mask；
- 从 `r_t + gamma V_{t+1} - V_t` 推导 GAE 和 Return；
- 实现 Probability Ratio、PPO-Clip、Value Loss、Entropy 和 KL；
- 使用共享 Backbone + Value Head 的轻量 Actor-Critic 结构；
- 记录行为策略版本、Reward、Advantage、Clip Fraction 和 Explained Variance。

## 1. PPO 训练协议

同步串行流程为：`Actor Rollout → 保存 Token/Logprob/Mask → Verifier Reward → Critic Value → GAE → PPO Update → 保存 Adapter`。vLLM 和训练模型不同时驻留 GPU。本轮仅训练 Equation Numeric；Gravity 与 Roman 只做能力保持评测。

核心目标：

\[
r_t(\theta)=\exp(\log\pi_\theta-\log\pi_{old}),\quad
L_{clip}=-E[\min(r_t A_t,\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t)]
\]

\[
\delta_t=r_t+\gamma(1-d_t)V_{t+1}-V_t,\quad
A_t=\delta_t+\gamma\lambda(1-d_t)A_{t+1}
\]

In [ ]:
from __future__ import annotations

import json
import math
import os
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd()
RESULT_DIR = PROJECT_ROOT / "data" / "processed" / "ppo"
CHECKPOINT_ROOT = PROJECT_ROOT / "data" / "processed" / "checkpoints"
SFT_MANIFEST_PATH = CHECKPOINT_ROOT / "sft_manifest.json"
TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "splits" / "train.csv"
MODEL_ID = os.environ.get("QWEN3_MODEL_ID", "Qwen/Qwen3-0.6B")
MODEL_TAG = MODEL_ID.rsplit("/", 1)[-1].lower().replace(".", "_")
DEFAULT_MODEL_PATH = Path.cwd() / "models" / MODEL_ID.rsplit("/", 1)[-1]
if not DEFAULT_MODEL_PATH.is_dir():
    DEFAULT_MODEL_PATH = Path.home() / "autodl-tmp/huggingface/models" / MODEL_ID.rsplit("/", 1)[-1]
MODEL_SOURCE = os.environ.get("QWEN3_MODEL_PATH", os.environ.get("LOCAL_MODEL_PATH",
    str(DEFAULT_MODEL_PATH) if DEFAULT_MODEL_PATH.is_dir() else MODEL_ID))
# Catch a stale 4B path before recording it under the 0.6B model ID.
if MODEL_ID == "Qwen/Qwen3-0.6B" and Path(MODEL_SOURCE).is_dir():
    _local_config = json.loads((Path(MODEL_SOURCE) / "config.json").read_text())
    if (_local_config.get("model_type"), _local_config.get("hidden_size"), _local_config.get("num_hidden_layers")) != ("qwen3", 1024, 28):
        raise ValueError("QWEN3_MODEL_PATH 与 Qwen3-0.6B 不匹配；请检查旧模型环境变量")
BASE_MODEL_PATH = MODEL_SOURCE
RESULT_DIR = RESULT_DIR / MODEL_TAG / os.environ.get("PPO_RUN_ID", os.environ.get("GPU_SMOKE_RUN_ID", ""))
CHECKPOINT_ROOT = CHECKPOINT_ROOT / MODEL_TAG / os.environ.get("GPU_SMOKE_RUN_ID", "")
SFT_MANIFEST_PATH = Path(os.environ.get("PPO_SFT_MANIFEST", str(CHECKPOINT_ROOT / "sft_manifest.json"))).resolve()
CHECKPOINT_ROOT = PROJECT_ROOT / "data/processed/checkpoints" / MODEL_TAG / os.environ.get("PPO_RUN_ID", os.environ.get("GPU_SMOKE_RUN_ID", ""))
SEED = 42
RUN_PPO = os.environ.get("RUN_PPO", "0") == "1"  # umbrella flag
RUN_PPO_ROLLOUT = os.environ.get("RUN_PPO_ROLLOUT", "1" if RUN_PPO else "0") == "1"
RUN_PPO_TRAINING = os.environ.get("RUN_PPO_TRAINING", "1" if RUN_PPO else "0") == "1"
RUN_MODE = os.environ.get("RUN_MODE", "smoke")
GROUP_TASKS = ("equation_numeric",)
SMOKE_PROMPTS_PER_TASK = 2
MAX_NEW_TOKENS = int(os.environ.get("RL_MAX_NEW_TOKENS", "16384"))
MAX_MODEL_LEN = int(os.environ.get("RL_MAX_MODEL_LEN", "20480"))
# T=1/top-p=1 makes the behavior distribution explicit for the first alignment test.
TEMPERATURE = 1.0
TOP_P = 1.0
TRAINING_PROFILE = "official-q-qlora"  # 必须具备 bitsandbytes；vLLM 使用独立 Profile
PPO_MICRO_BATCH = 1
TOKEN_CHUNK_SIZE = int(os.environ.get("PPO_TOKEN_CHUNK_SIZE", "128"))
RUN_PPO_MEMORY_TEST = os.environ.get("RUN_PPO_MEMORY_TEST", "0") == "1"
MEMORY_TEST_SEQUENCE_LENGTH = 16384  # 不随 Rollout 预算静默缩短
VLLM_MAX_NUM_SEQS = int(os.environ.get("VLLM_MAX_NUM_SEQS", "2"))
VLLM_KV_CACHE_BYTES = int(float(os.environ.get("VLLM_KV_CACHE_GIB", "4")) * 2**30)
VLLM_MAX_BATCHED_TOKENS = int(os.environ.get("VLLM_MAX_BATCHED_TOKENS", "1024"))
if min(VLLM_MAX_NUM_SEQS, VLLM_KV_CACHE_BYTES, VLLM_MAX_BATCHED_TOKENS) < 1:
    raise ValueError("Memory profile values must be positive")
if TOKEN_CHUNK_SIZE < 1: raise ValueError("TOKEN_CHUNK_SIZE must be positive")
GAMMA = 1.0
GAE_LAMBDA = 1.0
PPO_CLIP_EPS = 0.2
VALUE_COEF = 0.5
ENTROPY_COEF = 0.01
KL_COEF = 0.0
LEARNING_RATE = 1e-5
PPO_ROLLOUT_PATH = RESULT_DIR / "ppo_smoke_rollouts.jsonl"
PPO_ROLLOUT_MANIFEST_PATH = RESULT_DIR / "ppo_rollout_manifest.json"
PPO_TRAINING_MANIFEST_PATH = RESULT_DIR / "ppo_training_manifest.json"

np.random.seed(SEED)
torch.manual_seed(SEED)
print({"model_id": MODEL_ID, "run_ppo": RUN_PPO, "run_mode": RUN_MODE, "max_new_tokens": MAX_NEW_TOKENS, "tasks": GROUP_TASKS})


## 2. Rollout 数据结构与 Mask 约定

`input_ids` 包含 Prompt 和 Completion；`response_mask=1` 只覆盖 Completion Token，EOS 属于 Completion，Padding 为 0。`old_logprobs` 和 `old_values` 必须来自生成这批 Rollout 的行为策略版本，不能用更新后的 Policy 重新计算后冒充 Old Policy。

In [ ]:
@dataclass
class PPORollout:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    response_mask: torch.Tensor
    old_logprobs: torch.Tensor
    old_values: torch.Tensor
    rewards: torch.Tensor
    dones: torch.Tensor
    policy_version: str
    verifier_version: str
    finish_reason: str
    truncated: bool
    task_type: str
    sample_id: str
    source: str = "unknown"
    verified_correct: bool = False


def validate_rollout_alignment(rollout: PPORollout) -> None:
    shapes = [rollout.input_ids.shape, rollout.attention_mask.shape, rollout.response_mask.shape, rollout.old_logprobs.shape, rollout.old_values.shape, rollout.rewards.shape, rollout.dones.shape]
    if len({tuple(shape) for shape in shapes}) != 1:
        raise ValueError(f"rollout tensors are misaligned: {shapes}")
    if rollout.input_ids.ndim != 2:
        raise ValueError("rollout tensors must have shape [B,T]")
    if not torch.all((rollout.response_mask == 0) | (rollout.response_mask == 1)):
        raise ValueError("response_mask must be binary")
    if not torch.all((rollout.attention_mask == 0) | (rollout.attention_mask == 1)):
        raise ValueError("attention_mask must be binary")
    if not torch.all(rollout.response_mask <= rollout.attention_mask):
        raise ValueError("response_mask cannot include padding")
    if not torch.all(rollout.dones <= rollout.response_mask):
        raise ValueError("done can only occur on a response token")
    if torch.any(rollout.dones.sum(dim=1) > 1):
        raise ValueError("each sequence can have at most one observed EOS")
    if rollout.truncated and torch.any(rollout.dones > 0):
        raise ValueError("a length-truncated rollout cannot claim an observed EOS")


def outcome_reward(correct: bool | torch.Tensor, response_mask: torch.Tensor, task_type: str) -> torch.Tensor:
    """Put the binary outcome reward on each row's final valid Response token."""
    if response_mask.ndim == 1:
        response_mask = response_mask.unsqueeze(0)
    reward = torch.zeros_like(response_mask, dtype=torch.float32)
    correct_values = torch.as_tensor(correct, dtype=torch.bool, device=response_mask.device).reshape(-1)
    if correct_values.numel() == 1 and response_mask.size(0) > 1:
        correct_values = correct_values.expand(response_mask.size(0))
    if correct_values.numel() != response_mask.size(0):
        raise ValueError("correct must have one value per rollout row")
    for row_index in range(response_mask.size(0)):
        active_positions = torch.nonzero(response_mask[row_index].bool(), as_tuple=False).flatten()
        if active_positions.numel():
            reward[row_index, int(active_positions[-1])] = float(correct_values[row_index])
    return reward


def build_cpu_rollout(batch_size: int = 2, time_steps: int = 7, vocab_size: int = 32) -> PPORollout:
    torch.manual_seed(SEED)
    input_ids = torch.randint(0, vocab_size, (batch_size, time_steps))
    attention_mask = torch.ones(batch_size, time_steps, dtype=torch.long)
    response_mask = torch.zeros(batch_size, time_steps, dtype=torch.long)
    response_mask[:, 2:] = 1
    old_logprobs = torch.randn(batch_size, time_steps) * 0.1
    old_values = torch.randn(batch_size, time_steps) * 0.1
    rewards = torch.zeros(batch_size, time_steps)
    rewards[:, -1] = torch.tensor([1.0, 0.0])
    dones = torch.zeros(batch_size, time_steps)
    dones[:, -1] = 1.0
    rollout = PPORollout(input_ids, attention_mask, response_mask, old_logprobs, old_values, rewards, dones, "cpu_fixture", "toy_verifier", "stop", False, "equation_numeric", "cpu-ppo")
    validate_rollout_alignment(rollout)
    return rollout

### 维护者辅助函数：控制词表计算显存

概率与 Entropy 使用 FP32 的 LogSumExp 等价公式，沿序列每 128 Token 重算。训练时连 LM Head 投影一起分块，避免 16K 全词表张量；所有 Chunk 按全局有效 Token 数归约。Value Head 保持 FP32，GAE 和 PPO 目标不变。这些函数随参考实现提供，不作为学生练习。

In [ ]:
from torch.utils.checkpoint import checkpoint


def require_nf4_manifest(manifest):
    expected = {"mode":"nf4_qlora", "load_in_4bit":True, "quant_type":"nf4", "double_quant":True, "compute_dtype":"bfloat16"}
    if any(manifest.get("quantization", {}).get(k) != v for k,v in expected.items()):
        raise ValueError("SFT Manifest must specify NF4 + Double Quant + BF16 Compute; no training fallback")


def assert_nf4_training_model(model):
    base = model.base_model.get_base_model()
    quant = base.config.quantization_config
    quant = quant.to_dict() if hasattr(quant, "to_dict") else quant
    checks = {"loaded_in_4bit":bool(getattr(base,"is_loaded_in_4bit",False)),
              "nf4":quant.get("bnb_4bit_quant_type")=="nf4",
              "double_quant":quant.get("bnb_4bit_use_double_quant") is True,
              "bf16_compute":str(quant.get("bnb_4bit_compute_dtype")).replace("torch.","")=="bfloat16",
              "gradient_checkpointing":bool(base.is_gradient_checkpointing),
              "use_cache_disabled":base.config.use_cache is False,
              "value_head_fp32":model.value_head.weight.dtype==torch.float32}
    if not all(checks.values()): raise ValueError(f"PPO runtime NF4 contract failed: {checks}")
    return checks


def _recompute(function, *args):
    if torch.is_grad_enabled() and any(isinstance(x, torch.Tensor) and x.requires_grad for x in args):
        return checkpoint(function, *args, use_reentrant=False)
    return function(*args)


def _effective_attention_mask(mask):
    # Fully valid sequences need only the model causal mask; preserve real padding.
    # BF16 autocast at NF4 forward sites is separately required for efficient SDPA.
    return None if mask is None or bool(torch.all(mask == 1)) else mask


def _selected_logprob(logits, labels):
    z = logits.float()
    return z.gather(-1, labels.unsqueeze(-1)).squeeze(-1) - torch.logsumexp(z, dim=-1)


def _token_entropy(logits):
    z = logits.float()
    log_z = torch.logsumexp(z, dim=-1)
    return log_z - (torch.exp(z-log_z.unsqueeze(-1)) * z).sum(-1)


def token_logprobs(logits, input_ids, chunk_size=TOKEN_CHUNK_SIZE):
    """logits[t] predicts token[t+1]; never allocate full FP32 log-softmax."""
    if logits.shape[:2] != input_ids.shape or input_ids.size(1) < 2: raise ValueError("Token/label alignment error")
    if chunk_size < 1: raise ValueError("Invalid chunk size")
    return torch.cat([_recompute(_selected_logprob, logits[:,s:min(s+chunk_size, logits.size(1)-1)],
                                input_ids[:,s+1:min(s+chunk_size, logits.size(1)-1)+1])
                      for s in range(0, logits.size(1)-1, chunk_size)], dim=1)


def entropy_from_logits(logits, response_mask, chunk_size=TOKEN_CHUNK_SIZE):
    if logits.shape[:2] != response_mask.shape: raise ValueError("Entropy mask alignment error")
    if chunk_size < 1: raise ValueError("Invalid chunk size")
    # Normalize ONCE by global valid-token count, never mean of chunk means.
    total = logits.new_zeros((), dtype=torch.float32)
    for s in range(0, logits.size(1), chunk_size):
        block = _recompute(_token_entropy, logits[:,s:s+chunk_size])
        total = total + (block * response_mask[:,s:s+chunk_size]).sum()
    return total / response_mask.sum().clamp_min(1e-8)


def projected_policy_statistics(hidden, lm_head, labels, response_mask, chunk_size=TOKEN_CHUNK_SIZE):
    """Checkpoint projection as well: dense [B,T,V] logits never exist in Training."""
    if hidden.shape[:2] != labels.shape or labels.shape != response_mask.shape: raise ValueError("Projected action axis mismatch")
    def block_statistics(h, y):
        logits = lm_head(h)
        return _selected_logprob(logits, y), _token_entropy(logits)
    logprobs, entropy_sum = [], hidden.new_zeros((), dtype=torch.float32)
    for s in range(0, hidden.size(1), chunk_size):
        lp, h = _recompute(block_statistics, hidden[:,s:s+chunk_size], labels[:,s:s+chunk_size])
        logprobs.append(lp)
        entropy_sum = entropy_sum + (h * response_mask[:,s:s+chunk_size]).sum()
    return torch.cat(logprobs, dim=1), entropy_sum / response_mask.sum().clamp_min(1e-8)

## 3. GAE、Return、PPO-Clip 和诊断指标

Padding 与终止状态都通过 Mask 处理。终止位置的 bootstrap 被关闭；Padding 不贡献 Advantage、Value Loss、Entropy 或 KL。`advantages` 只在有效 Response Token 上标准化。

In [ ]:
def masked_mean(values: torch.Tensor, mask: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    mask = mask.to(values.dtype)
    return (values * mask).sum() / mask.sum().clamp_min(eps)


def compute_gae(rewards: torch.Tensor, values: torch.Tensor, dones: torch.Tensor, response_mask: torch.Tensor, gamma: float = GAMMA, lam: float = GAE_LAMBDA) -> tuple[torch.Tensor, torch.Tensor]:
    if rewards.shape != values.shape or rewards.shape != dones.shape or rewards.shape != response_mask.shape:
        raise ValueError("rewards, values, dones and response_mask must share [B,T]")
    batch, time_steps = rewards.shape
    mask = response_mask.to(rewards.dtype)
    advantages = torch.zeros_like(rewards)
    next_advantage = torch.zeros(batch, device=rewards.device, dtype=rewards.dtype)
    for t in reversed(range(time_steps)):
        valid = mask[:, t]
        next_valid = mask[:, t + 1] if t + 1 < time_steps else torch.zeros_like(valid)
        nonterminal = (1.0 - dones[:, t].to(rewards.dtype)) * next_valid
        next_value = values[:, t + 1] if t + 1 < time_steps else torch.zeros_like(values[:, t])
        delta = (rewards[:, t] + gamma * next_value * nonterminal - values[:, t]) * valid
        next_advantage = delta + gamma * lam * nonterminal * next_advantage
        next_advantage = next_advantage * valid
        advantages[:, t] = next_advantage
    returns = (advantages + values) * mask
    return advantages, returns


def normalize_masked(values: torch.Tensor, mask: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    mean = masked_mean(values, mask)
    variance = masked_mean((values - mean) ** 2, mask)
    return ((values - mean) / torch.sqrt(variance + eps)) * mask.to(values.dtype)


def ppo_clip_loss(old_logprobs: torch.Tensor, new_logprobs: torch.Tensor, advantages: torch.Tensor, response_mask: torch.Tensor, clip_eps: float = PPO_CLIP_EPS) -> dict[str, torch.Tensor]:
    ratio = torch.exp(new_logprobs - old_logprobs)
    if not torch.isfinite(ratio).all():
        raise ValueError("Nonfinite policy ratio; refuse silent gradient clipping in log space")
    unclipped = ratio * advantages
    clipped_ratio = ratio.clamp(1.0 - clip_eps, 1.0 + clip_eps)
    clipped = clipped_ratio * advantages
    objective = torch.minimum(unclipped, clipped)
    clip_fraction = ((ratio - clipped_ratio).abs() > 1e-8).to(torch.float32)
    return {
        "policy_loss": -masked_mean(objective, response_mask),
        "ratio": ratio.detach(),
        "clip_fraction": masked_mean(clip_fraction, response_mask),
    }


def value_loss(values: torch.Tensor, returns: torch.Tensor, response_mask: torch.Tensor) -> torch.Tensor:
    return masked_mean((values - returns.detach()) ** 2, response_mask)


def approx_kl(old_logprobs: torch.Tensor, new_logprobs: torch.Tensor, response_mask: torch.Tensor) -> torch.Tensor:
    return masked_mean(old_logprobs - new_logprobs, response_mask)


def explained_variance(values: torch.Tensor, returns: torch.Tensor, response_mask: torch.Tensor) -> torch.Tensor:
    mask = response_mask.bool()
    y, y_pred = returns.detach()[mask], values.detach()[mask]
    if y.numel() < 2 or torch.var(y) < 1e-8:
        return torch.tensor(0.0, device=values.device)
    return 1.0 - torch.var(y - y_pred) / torch.var(y)


## 4. 共享 Backbone + Scalar Value Head

真实 Qwen3 模型由 Transformers 加载，Value Head 读取最后一层 Hidden State；Policy Logits 仍来自 Causal LM Head。第二轮 Runner 使用 BF16 LoRA；LoRA 参数和 FP32 Value Head 共享 Backbone 并参与更新。历史 standalone NF4 加载器仍保留，但不是第二轮正式模式。这里的 Tiny 模型用于 CPU 机制测试。

**时间轴约定（本轮审查修订）**：隐藏状态位置 `t` 是预测 token `t+1` 的动作前状态；因此 Policy Logprob 使用 `logits[:, :-1]`，Value 使用 `values[:, :-1]`，Reward、Done 和 Response Mask 使用 token 位置 `[:, 1:]`。Padding、Prompt 和截断后的不可见状态不参与 GAE 与 Value Loss。


In [ ]:
class TinyActorCritic(nn.Module):
    def __init__(self, vocab_size: int = 32, hidden_size: int = 24):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.trunk = nn.Sequential(nn.Linear(hidden_size, hidden_size), nn.Tanh())
        self.policy_head = nn.Linear(hidden_size, vocab_size)
        self.value_head = nn.Linear(hidden_size, 1)
    def forward(self, input_ids, attention_mask=None):
        hidden = self.trunk(self.embedding(input_ids))
        return {"logits": self.policy_head(hidden), "values": self.value_head(hidden).squeeze(-1)}


class CausalLMValueHead(nn.Module):
    """Value at full-sequence position t estimates the state before token t+1.

    The scalar head is deliberately colocated with the backbone and kept in
    FP32. Hidden states are cast to the head dtype before the matmul, so a
    BF16/NF4 base model never receives a mismatched-device or mismatched-dtype
    linear operation.
    """
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        hidden_size = int(base_model.config.hidden_size)
        base_parameter = next(base_model.parameters())
        self.value_head = nn.Linear(hidden_size, 1, bias=False).to(device=base_parameter.device, dtype=torch.float32)
    def hidden_and_values(self, input_ids, attention_mask=None):
        # Qwen3ForCausalLM.model retains injected PEFT LoRA layers. No causal-LM
        # projection and no output_hidden_states tuple are constructed here.
        causal = self.base_model.get_base_model()
        hidden = causal.model(input_ids=input_ids, attention_mask=_effective_attention_mask(attention_mask),
                              output_hidden_states=False, use_cache=False, return_dict=True).last_hidden_state
        self.last_hidden_dtype = str(hidden.dtype)
        # Value Head stays FP32 even inside BF16 autocast.
        with torch.autocast(device_type=hidden.device.type, enabled=False):
            values = self.value_head(hidden.float()).squeeze(-1)
        return hidden, values

    def forward(self, input_ids, attention_mask=None, response_mask=None):
        if hasattr(self.base_model, "get_base_model"):
            hidden, values = self.hidden_and_values(input_ids, attention_mask)
            if response_mask is not None:
                logprobs, entropy = projected_policy_statistics(
                    hidden[:, :-1], self.base_model.get_output_embeddings(),
                    input_ids[:, 1:], response_mask[:, 1:])
                return {"logprobs": logprobs, "entropy": entropy, "values": values}
            return {"logits": self.base_model.get_output_embeddings()(hidden), "values": values}
        # CPU dtype fixture and small dense reference only.
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True, use_cache=False)
        self.last_hidden_dtype = str(outputs.hidden_states[-1].dtype)
        hidden = outputs.hidden_states[-1].to(device=self.value_head.weight.device, dtype=torch.float32)
        return {"logits": outputs.logits, "values": self.value_head(hidden).squeeze(-1)}

    def values_only(self, input_ids, attention_mask=None):
        return self.hidden_and_values(input_ids, attention_mask)[1]


def align_response_mask(response_mask: torch.Tensor) -> torch.Tensor:
    # The token at position t is predicted by logits at position t-1.
    return response_mask[:, 1:].to(torch.float32)


def align_ppo_action_axis(
    rewards: torch.Tensor,
    old_values: torch.Tensor,
    dones: torch.Tensor,
    response_mask: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Align all rollout fields to actions predicting token positions 1..T-1."""
    shapes = {tuple(value.shape) for value in (rewards, old_values, dones, response_mask)}
    if len(shapes) != 1 or rewards.ndim != 2 or rewards.size(1) < 2:
        raise ValueError("PPO full-sequence tensors must share [B,T] with T>=2")
    # Hidden state t predicts token t+1; reward/done/mask live on token t+1.
    return rewards[:, 1:], old_values[:, :-1], dones[:, 1:], response_mask[:, 1:].to(torch.float32)


## 5. 一次 PPO Update

Old Log Probability 和 Old Value 由 Rollout 固定；更新时只重新计算当前 Actor/Critic。总 Loss 为 Policy Loss + Value Loss − Entropy；本实现仅记录 Old Policy 概率漂移诊断，没有 Reference KL；`KL_COEF` 非0直接报错。

In [ ]:
def ppo_update(model: nn.Module, optimizer: torch.optim.Optimizer, rollout: PPORollout, clip_eps: float = PPO_CLIP_EPS, value_coef: float = VALUE_COEF, entropy_coef: float = ENTROPY_COEF, kl_coef: float = KL_COEF, step_optimizer: bool = True, loss_scale: float = 1.0, advantage_stats=None) -> dict[str, float]:
    if kl_coef != 0:
        raise ValueError("Reference KL is not implemented; old-policy drift is diagnostic only")
    validate_rollout_alignment(rollout)
    model.train()
    for module in model.modules():
        if isinstance(module, nn.Dropout): module.eval()
    started = time.perf_counter()
    if rollout.input_ids.is_cuda: torch.cuda.synchronize()
    if isinstance(model, CausalLMValueHead) and hasattr(model.base_model, "get_base_model"):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(rollout.input_ids, rollout.attention_mask, response_mask=rollout.response_mask)
    else:
        outputs = model(rollout.input_ids, rollout.attention_mask)
    values = outputs["values"]
    new_logprobs = outputs["logprobs"] if "logprobs" in outputs else token_logprobs(outputs["logits"], rollout.input_ids)  # [B,T-1], actions for token positions 1..T-1
    old_logprobs = rollout.old_logprobs[:, 1:]
    action_rewards, action_old_values, action_dones, action_mask = align_ppo_action_axis(
        rollout.rewards, rollout.old_values, rollout.dones, rollout.response_mask
    )
    advantages, returns = compute_gae(action_rewards, action_old_values, action_dones, action_mask)
    if advantage_stats is None:
        advantages = normalize_masked(advantages, action_mask)
    else:
        mean, variance = advantage_stats
        advantages = ((advantages - mean) / torch.sqrt(variance + 1e-8)) * action_mask
    policy = ppo_clip_loss(old_logprobs, new_logprobs, advantages, action_mask, clip_eps)
    # The value at hidden position t estimates the state before action token t+1.
    current_action_values = values[:, :-1]
    value = value_loss(current_action_values, returns, action_mask)
    entropy = outputs["entropy"] if "entropy" in outputs else entropy_from_logits(outputs["logits"][:, :-1, :], action_mask)
    kl = approx_kl(old_logprobs, new_logprobs, action_mask)
    total = policy["policy_loss"] + value_coef * value - entropy_coef * entropy + kl_coef * kl
    if not torch.isfinite(total): raise ValueError("Non-finite PPO objective")
    if step_optimizer: optimizer.zero_grad(set_to_none=True)
    if rollout.input_ids.is_cuda: torch.cuda.synchronize()
    forward_seconds = time.perf_counter() - started
    backward_started = time.perf_counter()
    (total * loss_scale).backward()
    if rollout.input_ids.is_cuda: torch.cuda.synchronize()
    backward_seconds = time.perf_counter() - backward_started
    if step_optimizer:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    return {
        "forward_seconds": forward_seconds, "backward_seconds": backward_seconds,
        "total_loss": float(total.detach()),
        "policy_loss": float(policy["policy_loss"].detach()),
        "value_loss": float(value.detach()),
        "entropy": float(entropy.detach()),
        "approx_kl": float(kl.detach()),
        "clip_fraction": float(policy["clip_fraction"].detach()),
        "advantage_mean": float(masked_mean(advantages, action_mask).detach()),
        "advantage_std": float(torch.sqrt(masked_mean(advantages ** 2, action_mask)).detach()),
        "explained_variance": float(explained_variance(current_action_values, returns, action_mask).detach()),
        "action_tokens": float(action_mask.sum().detach()),
    }


## 6. GPU Rollout / PPO 入口（当前关闭）

真实入口使用 vLLM 生成并保留行为策略 Logprobs；如果 vLLM Profile 无法返回所需 Logprobs，必须停止并记录兼容性问题，不能用当前 Policy Logprobs 冒充 Old Policy。训练完成后保存 Adapter，再串行切回 vLLM。

### 6.1 读取并校验 SFT Manifest

In [ ]:
def read_sft_manifest(path: Path = SFT_MANIFEST_PATH) -> dict[str, object]:
    if not path.exists():
        raise FileNotFoundError(f"SFT Manifest 不存在：{path}；请先运行 Notebook 03")
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if manifest.get("schema_version") != "nemotron-sft-checkpoint/v1" or manifest.get("base_model", {}).get("id") != MODEL_ID:
        raise ValueError("SFT Manifest schema 或 Base Model 不匹配")
    if manifest.get("status") != "completed":
        raise ValueError("SFT Manifest 尚未标记为 completed，不能初始化 PPO")
    require_nf4_manifest(manifest)
    adapter_path = Path(resolve_manifest_path(str(manifest.get("adapter_path", ""))))
    if not adapter_path.is_dir() or not (adapter_path / "adapter_config.json").exists():
        raise FileNotFoundError(f"SFT Adapter 不存在或缺少 adapter_config.json：{adapter_path}")
    return manifest


def resolve_manifest_path(value: str) -> str:
    path = Path(value)
    return str(path if path.is_absolute() else PROJECT_ROOT / path)


### 6.2 解析 vLLM 行为策略 Logprob与任务答案验证

In [ ]:


OFFICIAL_VERIFIER_VERSION = "official_nvidia_metric_v3"


def _normalize_answer_text(value: object) -> str:
    text = str(value).strip()
    return __import__("re").sub(r"^```(?:text|json)?\s*|\s*```$", "", text, flags=__import__("re").I).strip()


def _extract_rollout_answer(text: str, task_type: str) -> tuple[str | None, str]:
    """Extract an explicitly marked final answer; never use an arbitrary CoT number."""
    import re
    text = str(text)
    boxed = re.findall(r"\\boxed\{([^{}]+)\}", text)
    if boxed:
        return _normalize_answer_text(boxed[-1]), "boxed"
    tagged = re.findall(r"<(?P<tag>final|answer)>\s*(?P<value>.*?)\s*</(?P=tag)>", text, flags=re.I | re.S)
    if tagged:
        return _normalize_answer_text(tagged[-1][1]), "tagged"
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in reversed(lines):
        labeled = re.match(r"(?i)^(?:final answer|answer|最终答案|最终输出)\s*[:：]\s*(.+)$", line)
        if labeled:
            candidate = _normalize_answer_text(labeled.group(1)).strip("`$")
            if task_type == "bit_manipulation" and re.fullmatch(r"(?:0b)?[01]{8}", candidate):
                return candidate.removeprefix("0b"), "final_label"
            if task_type in {"gravity", "equation_numeric"} and re.fullmatch(r"[-+]?\d+(?:\.\d+)?", candidate):
                return candidate, "final_label"
        if task_type == "bit_manipulation" and re.fullmatch(r"(?:0b)?[01]{8}", line):
            return line.removeprefix("0b"), "standalone_final_line"
        if task_type in {"gravity", "equation_numeric"} and re.fullmatch(r"[-+]?\d+(?:\.\d+)?", line):
            return line, "standalone_final_line"
    return None, "unmarked_or_empty"


def _verify_rollout(task_type: str, completion: str, expected: str, truncated: bool = False) -> tuple[bool, str]:
    if truncated:
        return False, "truncated_output"
    predicted, _ = _extract_rollout_answer(completion, task_type)
    if predicted is None:
        return False, "not_extracted"
    predicted_text, expected_text = _normalize_answer_text(predicted), _normalize_answer_text(expected)
    if task_type == "bit_manipulation":
        p, e = predicted_text.removeprefix("0b"), expected_text.removeprefix("0b")
        if not __import__("re").fullmatch(r"[01]{8}", p) or not __import__("re").fullmatch(r"[01]{8}", e):
            return False, "invalid_8bit_format"
        return p == e, "exact_8bit"
    if task_type in {"gravity", "equation_numeric"}:
        if not __import__("re").fullmatch(r"[-+]?\d+(?:\.\d+)?", predicted_text):
            return False, "invalid_numeric_format"
        try:
            ok = math.isclose(float(predicted_text), float(expected_text), rel_tol=1e-2, abs_tol=1e-5)
        except (TypeError, ValueError):
            ok = False
        return ok, "official_math_isclose"
    return predicted_text == expected_text, "exact_text"


### 6.3 vLLM Rollout：保留 Token IDs、Logprob 和 EOS 状态

In [ ]:
def run_vllm_ppo_rollout(prompt_rows, model_source, policy_version, adapter_path=None):
    records = _generate_rollout_records(prompt_rows, model_source or BASE_MODEL_PATH, policy_version, adapter_path, 1)
    for row in records:
        row["correct"], row["verification_method"] = _verify_rollout(row["task_type"], row["completion"], row["expected_answer"], row["truncated"])
        row["verifier_version"] = OFFICIAL_VERIFIER_VERSION
    return records


### 6.4 从 SFT Adapter 加载 Actor-Critic

In [ ]:
def _load_sft_actor_critic(manifest: dict, mode: str = "nf4_qlora", value_head_path: Path | None = None, *, audit_only: bool = False):
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel, prepare_model_for_kbit_training
    require_nf4_manifest(manifest)
    if mode != "nf4_qlora" and not (audit_only and mode == "bf16_lora"):
        raise ValueError("PPO Training requires NF4 QLoRA; BF16 is read-only audit only")
    kwargs = {"device_map":{"":0}, "torch_dtype":torch.bfloat16, "attn_implementation":"sdpa"}
    if mode == "nf4_qlora":
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    base = AutoModelForCausalLM.from_pretrained(os.environ.get("QWEN3_MODEL_PATH", manifest["base_model"]["path"]), **kwargs)
    base.config.use_cache = False
    if mode == "nf4_qlora":
        base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True,
                gradient_checkpointing_kwargs={"use_reentrant":False})
        # prepare_model_for_kbit_training upcasts the frozen tied embedding/head.
        # Restore BF16 storage; probability reductions and Value Head stay FP32.
        base.get_input_embeddings().to(dtype=torch.bfloat16)
        base.get_output_embeddings().to(dtype=torch.bfloat16)
    actor = PeftModel.from_pretrained(base, resolve_manifest_path(manifest["adapter_path"]), is_trainable=not audit_only)
    if not audit_only:
        actor.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})
        actor.enable_input_require_grads()
    wrapped = CausalLMValueHead(actor)
    if value_head_path is not None:
        wrapped.value_head.load_state_dict(torch.load(value_head_path, map_location=wrapped.value_head.weight.device, weights_only=True))
    if audit_only:
        wrapped.requires_grad_(False)
    else:
        unexpected = [n for n,p in wrapped.named_parameters() if p.requires_grad and "lora_" not in n and not n.startswith("value_head.")]
        if unexpected: raise ValueError(f"Unexpected trainable parameters: {unexpected}")
        wrapped.nf4_contract = assert_nf4_training_model(wrapped)
    wrapped.eval()
    return wrapped


### 6.5 把 Rollout 转成 PPO 时间轴张量

In [ ]:
def build_ppo_rollout_from_records(records: list[dict[str, object]], actor_critic: nn.Module, pad_token_id: int = 0) -> PPORollout:
    if not records: raise ValueError("empty PPO rollout")
    if len(records) != PPO_MICRO_BATCH: raise ValueError("PPO micro-batch must be 1")
    sequences, prompt_lengths, completion_lengths, old_logprob_rows = [], [], [], []
    rewards_correct = []
    for record in records:
        prompt_ids, completion_ids = list(record["prompt_token_ids"]), list(record["completion_token_ids"])
        if not prompt_ids: raise ValueError("PPO rollout missing prompt_token_ids")
        if len(completion_ids) != len(record["behavior_logprobs"]): raise ValueError("PPO behavior Logprob alignment error")
        sequences.append(prompt_ids + completion_ids); prompt_lengths.append(len(prompt_ids)); completion_lengths.append(len(completion_ids)); old_logprob_rows.append([float(v) for v in record["behavior_logprobs"]]); rewards_correct.append(bool(record["correct"]))
    max_len = max(map(len, sequences)); batch = len(sequences)
    device = next(actor_critic.parameters()).device
    input_ids = torch.full((batch, max_len), int(pad_token_id), dtype=torch.long, device=device)
    attention_mask = torch.zeros((batch, max_len), dtype=torch.long, device=device)
    response_mask = torch.zeros((batch, max_len), dtype=torch.long, device=device)
    old_logprobs = torch.zeros((batch, max_len), dtype=torch.float32, device=device)
    dones = torch.zeros((batch, max_len), dtype=torch.float32, device=device)
    for row, (sequence, prompt_len, completion_len, old_values, record) in enumerate(zip(sequences, prompt_lengths, completion_lengths, old_logprob_rows, records)):
        input_ids[row, :len(sequence)] = torch.tensor(sequence, dtype=torch.long, device=device); attention_mask[row, :len(sequence)] = 1; response_mask[row, prompt_len:len(sequence)] = 1
        if old_values: old_logprobs[row, prompt_len:prompt_len + completion_len] = torch.tensor(old_values, dtype=torch.float32, device=device)
        if record.get("finish_reason") == "stop" and not record.get("truncated") and completion_len: dones[row, prompt_len + completion_len - 1] = 1
    rewards = outcome_reward(torch.tensor(rewards_correct, device=device), response_mask, str(records[0]["task_type"]))
    with torch.no_grad():
        actor_critic.eval()
        with torch.autocast("cuda", dtype=torch.bfloat16):
            old_values = actor_critic.values_only(input_ids, attention_mask).detach()
    rollout = PPORollout(input_ids, attention_mask, response_mask, old_logprobs, old_values, rewards, dones, str(records[0].get("policy_version", "unknown")), OFFICIAL_VERIFIER_VERSION, str(records[0].get("finish_reason", "unknown")), bool(any(record.get("truncated") for record in records)), str(records[0]["task_type"]), str(records[0]["sample_id"]))
    rollout.source = "vllm_rollout" if all(record.get("source")=="vllm_rollout" and not record.get("fixture") for record in records) else "unknown"
    rollout.verified_correct = all(bool(record.get("correct")) and not record.get("truncated") and record.get("finish_reason")=="stop" for record in records)
    validate_rollout_alignment(rollout)
    return rollout


### 6.6 保存 PPO Adapter、Value Head 与 Manifest

In [ ]:
def _save_ppo_update(model, logs: list[dict[str, float]], sft_manifest_path: Path, mode: str, parent_manifest: dict[str, object] | None = None) -> dict[str, object]:
    output_dir = CHECKPOINT_ROOT / f"ppo_{mode}"
    output_dir.mkdir(parents=True, exist_ok=True)
    model.base_model.save_pretrained(output_dir)
    torch.save(model.value_head.state_dict(), output_dir / "value_head.pt")
    parent_manifest = parent_manifest or read_sft_manifest(sft_manifest_path)
    ppo_manifest = {"schema_version":"nemotron-ppo-checkpoint/v1", "status":"pending_reload", "parent_sft_manifest":str(sft_manifest_path.relative_to(PROJECT_ROOT)), "parent_sft_adapter":str(parent_manifest["adapter_path"]), "base_model":parent_manifest["base_model"], "tokenizer_path":parent_manifest.get("tokenizer_path"), "adapter_path":str(output_dir.relative_to(PROJECT_ROOT)), "value_head_path":str((output_dir / "value_head.pt").relative_to(PROJECT_ROOT)), "mode":mode, "quantization":parent_manifest["quantization"], "micro_batch":1, "chunk_size":TOKEN_CHUNK_SIZE, "gradient_checkpointing":True, "use_cache":False, "policy_version":f"ppo:{mode}:seed{SEED}", "seed":SEED, "profile":TRAINING_PROFILE}
    (output_dir / "ppo_manifest.json").write_text(json.dumps(ppo_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    return {"status":"artifact_saved", "micro_steps":len(logs), "logs":logs, "adapter_path":str(output_dir), "value_head_path":str(output_dir / "value_head.pt"), "manifest_path":str(output_dir / "ppo_manifest.json")}


def _reload_ppo_artifact(parent_manifest, artifact, mode):
    """Explicit standalone Base+Adapter+ValueHead reload helper."""
    reloaded_manifest = dict(parent_manifest, adapter_path=str(artifact["adapter_path"]))
    model = _load_sft_actor_critic(reloaded_manifest, mode, value_head_path=Path(artifact["value_head_path"]))
    return model


### 6.7 PPO Update 调度

In [ ]:
def run_ppo_training(rollouts, sft_manifest_path=SFT_MANIFEST_PATH, mode="nf4_qlora", actor_critic=None):
    if mode != "nf4_qlora": raise ValueError("PPO Training requires NF4 QLoRA")
    if not RUN_PPO_TRAINING: return {"status":"skipped_by_default"}
    if not rollouts: return {"status":"blocked_empty_rollout", "optimizer_steps":0}
    if actor_critic is None: raise ValueError("PPO must use the SAME actor-critic that produced old_values")
    parent = read_sft_manifest(sft_manifest_path)
    for rollout in rollouts:
        if rollout.source != "vllm_rollout" or not rollout.verified_correct or rollout.policy_version != parent["policy_version"] or rollout.verifier_version != OFFICIAL_VERIFIER_VERSION:
            raise ValueError("PPO Update requires verified real Rollout from the exact parent SFT Policy")
        if rollout.input_ids.size(0) != 1: raise ValueError("PPO micro-batch must be 1")
        if rollout.truncated or rollout.finish_reason != "stop" or not rollout.rewards.sum() > 0 or "fixture" in rollout.policy_version:
            raise ValueError("Real PPO Update requires non-truncated, verifier-approved real Rollout")
    model = actor_critic
    assert_nf4_training_model(model)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE)
    active_advantages = []
    for rollout in rollouts:
        rewards, values, dones, mask = align_ppo_action_axis(rollout.rewards,rollout.old_values,rollout.dones,rollout.response_mask)
        advantage, _ = compute_gae(rewards,values,dones,mask)
        active_advantages.append(advantage[mask.bool()])
    all_advantages = torch.cat(active_advantages)
    stats = (all_advantages.mean(),all_advantages.var(unbiased=False))
    tokens = [int(r.response_mask[:,1:].sum()) for r in rollouts]
    optimizer.zero_grad(set_to_none=True)
    logs = [ppo_update(model,optimizer,r,step_optimizer=False,loss_scale=count/sum(tokens),advantage_stats=stats) for r,count in zip(rollouts,tokens)]
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    tracked = {name:p.detach().clone() for name,p in model.named_parameters() if p.requires_grad and ("lora_" in name or name.startswith("value_head."))}
    optimizer.step()
    _write_stage_json(RESULT_DIR/"ppo_update_progress.json", {"status":"optimizer_stepped_reload_pending", "optimizer_steps":1})
    deltas = {"lora":0.0,"value_head":0.0}
    for name,p in model.named_parameters():
        if name in tracked:
            key="value_head" if name.startswith("value_head.") else "lora"
            deltas[key]=max(deltas[key],float((p.detach()-tracked[name]).abs().max()))
    del tracked
    if not all(value>0 for value in deltas.values()): raise ValueError(f"Optimizer failed to change trainable parameters: {deltas}")
    manifest = read_sft_manifest(sft_manifest_path)
    artifact = _save_ppo_update(model, logs, sft_manifest_path, mode, parent_manifest=manifest)
    ids = rollouts[0].input_ids[0].detach().cpu().tolist()
    reload_check = actual_adapter_reload(model.base_model, Path(artifact["adapter_path"]), ids)
    # A separate FP32 module is loaded from disk and compared, not merely checked for existence.
    head = nn.Linear(model.value_head.in_features, 1, bias=False).to(device=model.value_head.weight.device, dtype=model.value_head.weight.dtype)
    head.load_state_dict(torch.load(artifact["value_head_path"], weights_only=True, map_location=model.value_head.weight.device))
    if not torch.equal(head.weight, model.value_head.weight): raise ValueError("PPO value head reload differs")
    metadata = json.loads(Path(artifact["manifest_path"]).read_text())
    metadata.update({"status":"completed", "optimizer_steps":1, "adapter_reload":reload_check, "value_head_reload":"passed", "micro_batch":1, "chunk_size":TOKEN_CHUNK_SIZE, "training_mode":"nf4_qlora", "nf4_runtime_contract":model.nf4_contract, "max_abs_parameter_update":deltas, "sequence_lengths":[int(r.input_ids.size(1)) for r in rollouts]})
    _write_stage_json(Path(artifact["manifest_path"]), metadata)
    return {**artifact, "status":"completed", "optimizer_steps":1, "adapter_reload":reload_check, "value_head_reload":"passed", "micro_batch":1, "chunk_size":TOKEN_CHUNK_SIZE, "training_mode":"nf4_qlora", "nf4_runtime_contract":model.nf4_contract, "max_abs_parameter_update":deltas, "sequence_lengths":[int(r.input_ids.size(1)) for r in rollouts]}


### 6.8 选择 Train Smoke Prompt

In [ ]:
def select_ppo_smoke_prompts(per_task: int = SMOKE_PROMPTS_PER_TASK) -> list[dict[str, object]]:
    if not TRAIN_PATH.exists(): raise FileNotFoundError("缺少 Train 划分")
    frame = pd.read_csv(TRAIN_PATH, dtype=str, keep_default_na=False)
    selected = []
    for task, offset in (("equation_numeric", 1), ("bit_manipulation", 2)):
        pool = frame[frame["task_type"] == task]
        for row in pool.sample(n=min(per_task, len(pool)), random_state=SEED + offset).itertuples(index=False):
            selected.append({"id":str(row.id), "task_type":str(row.task_type), "prompt":str(row.prompt), "answer":str(row.answer), "source_split":"train"})
    return selected


### 6.9 PPO Smoke Pipeline：Rollout → GAE → Update

In [ ]:
def _write_ppo_jsonl(path: Path, rows: list[dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(json.dumps(row, ensure_ascii=False) for row in rows) + ("\n" if rows else ""), encoding="utf-8")


def _read_ppo_jsonl(path: Path) -> list[dict[str, object]]:
    if not path.exists(): raise FileNotFoundError(f"阶段产物不存在：{path}")
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def _ppo_preflight(stage: str) -> dict[str, object]:
    if not torch.cuda.is_available(): return {"status":"blocked_no_cuda", "stage":stage}
    free, total = torch.cuda.mem_get_info()
    result = {"status":"ready", "stage":stage, "device":torch.cuda.get_device_name(0), "free_memory_mib":free/2**20, "total_memory_mib":total/2**20}
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    (RESULT_DIR / f"{stage}_preflight.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    return result


def run_ppo_rollout_phase():
    if not RUN_PPO_ROLLOUT: return {"status":"skipped_by_default"}
    def operation():
        sft = read_sft_manifest(); prompts = select_ppo_smoke_prompts()
        rows = run_vllm_ppo_rollout(prompts, sft["base_model"]["path"], sft["policy_version"], resolve_manifest_path(sft["adapter_path"]))
        return _persist_rollout(rows, PPO_ROLLOUT_PATH, PPO_ROLLOUT_MANIFEST_PATH, sft, len(prompts))
    return _gpu_stage("ppo_rollout", operation)


def run_ppo_training_phase(mode="nf4_qlora"):
    if mode != "nf4_qlora": raise ValueError("PPO Training requires NF4 QLoRA")
    if not RUN_PPO_TRAINING: return {"status":"skipped_by_default"}
    def operation():
        from transformers import AutoTokenizer
        import gc
        _write_stage_json(RESULT_DIR/"ppo_update_progress.json", {"status":"started", "optimizer_steps":0})
        rows = _read_rollout(PPO_ROLLOUT_PATH, PPO_ROLLOUT_MANIFEST_PATH)
        sft = read_sft_manifest()
        tokenizer = AutoTokenizer.from_pretrained(sft.get("tokenizer_path", sft["base_model"]["path"]))
        # First isolate engine/template differences on BF16, then measure NF4 drift.
        bf16 = _load_sft_actor_critic(sft, "bf16_lora", audit_only=True)
        baseline_alignment = audit_policy_logprobs(bf16.base_model, tokenizer, rows, "bf16_lora")
        del bf16; gc.collect(); torch.cuda.empty_cache()
        if baseline_alignment["status"] != "passed": return {**baseline_alignment, "optimizer_steps":0}
        model = _load_sft_actor_critic(sft, mode)
        alignment = audit_policy_logprobs(model.base_model, tokenizer, rows, mode)
        if alignment["status"] != "passed": return {**alignment, "optimizer_steps":0}
        usable = []
        for row in rows:
            row["correct"], row["verification_method"] = _verify_rollout(row["task_type"], row["completion"], row["expected_answer"], row["truncated"])
            if row["correct"] and not row["truncated"] and row["finish_reason"] == "stop": usable.append(row)
        for row in rows:
            row["verifier_version"] = OFFICIAL_VERIFIER_VERSION
            row["training_eligible"] = row in usable
            row["training_rejection_reasons"] = ([] if row in usable else
                (["truncated"] if row["truncated"] else []) + (["non_stop_finish"] if row["finish_reason"]!="stop" else []) + (["verifier_failed"] if not row["correct"] else []))
        _write_ppo_jsonl(RESULT_DIR / "ppo_verified_rollouts.jsonl", rows)
        if not usable:
            return {"status":"not_run_no_verified_rollout", "records":len(rows), "optimizer_steps":0, "alignment":alignment}
        # Keep the entire Smoke batch objective, but forward/backward one sequence at a time.
        pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
        rollouts = [build_ppo_rollout_from_records([row], model, int(pad)) for row in usable]
        result = run_ppo_training(rollouts, SFT_MANIFEST_PATH, mode, actor_critic=model)
        return {**result, "records":len(rows), "usable_records":len(usable), "ineligible_records":len(rows)-len(usable), "truncated_excluded":sum(bool(r["truncated"]) for r in rows), "verifier_failed_nontruncated":sum(not r["correct"] and not r["truncated"] for r in rows), "alignment":alignment}
    return _gpu_stage("ppo_training", operation)


def run_ppo_smoke_pipeline(mode="nf4_qlora"):
    if RUN_PPO_ROLLOUT and RUN_PPO_TRAINING:
        return {"status":"blocked_kernel_switch_required", "reason":"Run Rollout and Training in separate profiles/processes"}
    if RUN_PPO_ROLLOUT: return run_ppo_rollout_phase()
    if RUN_PPO_TRAINING: return run_ppo_training_phase(mode)
    return {"status":"skipped_by_default"}


## 7. CPU Smoke Test

Smoke Test 使用 Tiny Actor-Critic 和人工构造的张量，只验证 GAE 终止状态、Mask、Ratio、Clip Fraction、Value/Entropy/KL 和反向传播。它不是 Qwen3 的 PPO 结果，也不证明真实 Reward 有效。

In [ ]:
rollout = build_cpu_rollout()
action_rewards, action_old_values, action_dones, action_mask = align_ppo_action_axis(
    rollout.rewards, rollout.old_values, rollout.dones, rollout.response_mask
)
assert action_rewards.shape == (rollout.rewards.shape[0], rollout.rewards.shape[1] - 1)
assert torch.equal(action_mask, rollout.response_mask[:, 1:].float())
# Short-sequence alignment: token at position 2 uses state/value at position 1.
manual_values = torch.tensor([[99.0, 10.0, 20.0, 30.0]])
manual_rewards = torch.tensor([[0.0, 0.0, 0.0, 1.0]])
manual_dones = torch.tensor([[0.0, 0.0, 0.0, 1.0]])
manual_mask = torch.tensor([[0.0, 0.0, 1.0, 1.0]])
manual_r, manual_v, manual_d, manual_m = align_ppo_action_axis(
    manual_rewards, manual_values, manual_dones, manual_mask
)
assert torch.equal(manual_v, torch.tensor([[99.0, 10.0, 20.0]]))
assert torch.equal(manual_r, torch.tensor([[0.0, 0.0, 1.0]]))
assert torch.equal(manual_m, torch.tensor([[0.0, 1.0, 1.0]]))
advantages, returns = compute_gae(action_rewards, action_old_values, action_dones, action_mask)
assert advantages.shape == action_rewards.shape
assert returns.shape == action_rewards.shape
assert torch.all(advantages[:, :1] == 0)
assert torch.all(returns[:, :1] == 0)

model = TinyActorCritic()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)
logs = ppo_update(model, optimizer, rollout)
assert all(math.isfinite(value) for value in logs.values())
assert logs["clip_fraction"] >= 0.0
assert any(parameter.grad is not None for parameter in model.parameters())

# A terminal reward must not bootstrap through the next state.
terminal_rewards = torch.tensor([[0.0, 1.0]])
terminal_values = torch.tensor([[0.3, 0.4]])
terminal_dones = torch.tensor([[0.0, 1.0]])
terminal_mask = torch.ones_like(terminal_rewards)
terminal_adv, _ = compute_gae(terminal_rewards, terminal_values, terminal_dones, terminal_mask, gamma=0.9, lam=0.95)
expected_last = terminal_rewards[0, 1] - terminal_values[0, 1]
assert torch.isclose(terminal_adv[0, 1], expected_last)
# Padding after EOS must not propagate into the preceding response token.
padded_rewards = torch.tensor([[0.0, 1.0, 0.0, 0.0]])
padded_values = torch.tensor([[0.3, 0.4, 9.0, 9.0]])
padded_dones = torch.tensor([[0.0, 1.0, 0.0, 0.0]])
padded_mask = torch.tensor([[1.0, 1.0, 0.0, 0.0]])
padded_adv, padded_ret = compute_gae(padded_rewards, padded_values, padded_dones, padded_mask)
assert torch.all(padded_adv[:, 2:] == 0) and torch.all(padded_ret[:, 2:] == 0)

ppo_cpu_report = {
    "status": "passed",
    "gae_shape_and_mask": True,
    "value_action_alignment": True,
    "verifier_version": OFFICIAL_VERIFIER_VERSION,
    "terminal_bootstrap_disabled": True,
    "ppo_loss_finite": True,
    "policy_gradient": True,
    "clip_fraction": logs["clip_fraction"],
    "approx_kl": logs["approx_kl"],
    "gpu_rollout": "not_run",
    "gpu_training": "not_run",
}
print(json.dumps(ppo_cpu_report, ensure_ascii=False, indent=2))

## 本节总结与限制

PPO 的 GAE、PPO-Clip、Value Loss 与 Token 加权累积保持不变。维护者提供分块概率、Entropy 和 LM Head 重算；真实训练强制 NF4 QLoRA。

本轮 vGPU 已完成 16K 有效输入的真实模型前后向机制测试（Device Peak 15.49 GiB，无 Optimizer Step），以及两条合格真实 Rollout 的一次 PPO 更新（总长 7,009 / 7,185，Device Peak 11.49 GiB）。Adapter 与 FP32 Value Head 保存重载通过。16K 人工输入不属于真实 Rollout；未测更新后的准确率，不能据此声称能力提升。完整结果和失败诊断见 `docs/06_PPO.md` 第 15 节。


## 8. SFT Adapter 交接检查

PPO 只从 Notebook 03 的 SFT Manifest 加载 Base Model 和 PEFT Adapter；Value Head 单独保存。当前 CPU 阶段不加载模型。


In [ ]:
ppo_manifest_status = "available" if SFT_MANIFEST_PATH.exists() else "not_run_sft_manifest_missing"
if not (RUN_PPO_ROLLOUT or RUN_PPO_TRAINING):
    assert RUN_PPO is False
ppo_checkpoint_report = {"status":"passed", "smoke_prompts_per_task": SMOKE_PROMPTS_PER_TASK, "temperature": TEMPERATURE, "top_p": TOP_P, "sft_manifest": ppo_manifest_status, "gae_padding_mask_test": True, "value_action_alignment": True, "verifier_version": OFFICIAL_VERIFIER_VERSION, "gpu":"not_run"}
(RESULT_DIR / "cpu_ppo_smoke_report.json").parent.mkdir(parents=True, exist_ok=True)
(RESULT_DIR / "cpu_ppo_smoke_report.json").write_text(json.dumps(ppo_checkpoint_report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(ppo_checkpoint_report, ensure_ascii=False, indent=2))


### GPU 阶段：资源、失败与状态记录
Rollout 与 Training 使用独立进程和 Kernel；持久化完成后退出当前进程，再切换官方环境。


In [ ]:
def _write_stage_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")


def _gpu_stage(stage: str, operation) -> dict:
    """Preflight + device memory sampler + truthful failure/status report."""
    import gc, threading, subprocess, traceback, importlib.metadata
    started = time.perf_counter()
    stop, samples = threading.Event(), []
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    report_path = RESULT_DIR / f"{stage}_status.json"
    if stage=="ppo_training": _write_stage_json(RESULT_DIR/"ppo_update_progress.json", {"status":"stage_entered", "optimizer_steps":0})
    if not torch.cuda.is_available():
        result = {"status": "blocked_no_cuda", "stage": stage}
        _write_stage_json(report_path, result)
        return result
    free, total = torch.cuda.mem_get_info()
    preflight = {"device": torch.cuda.get_device_name(0), "total_mib": total / 2**20,
                 "free_mib": free / 2**20, "torch_cuda": torch.version.cuda,
                 "bf16_supported": torch.cuda.is_bf16_supported()}
    preflight["versions"] = {}
    for package in ("torch", "transformers", "peft", "bitsandbytes", "vllm"):
        try: preflight["versions"][package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: pass
    _write_stage_json(RESULT_DIR / f"{stage}_preflight.json", preflight)
    def monitor():
        while not stop.is_set():
            try:
                output = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True, timeout=3)
                samples.append(float(output.splitlines()[0]))
            except (OSError, ValueError, subprocess.SubprocessError): pass
            stop.wait(0.1)
    thread = threading.Thread(target=monitor, daemon=True)
    torch.cuda.reset_peak_memory_stats()
    thread.start()
    try:
        if ("training" in stage or "memory_mechanism" in stage) and "vllm" in __import__("sys").modules:
            result = {"status": "blocked_kernel_switch_required", "reason": "先结束 vLLM 进程并切换训练 Kernel"}
        elif free / total < 0.80:
            result = {"status": "blocked_gpu_occupied", "reason": "GPU 可用显存不足 80%；不终止其他任务"}
        else:
            result = operation()
            if not isinstance(result, dict) or "status" not in result:
                raise ValueError("阶段必须显式返回 status")
    except Exception as exc:
        message = traceback.format_exc()
        result = {"status": "oom" if isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in str(exc).lower() else "failed",
                  "error": repr(exc), "traceback": message}
        (RESULT_DIR / f"{stage}_error.log").write_text(message, encoding="utf-8")
    finally:
        stop.set(); thread.join(timeout=4)
        gc.collect(); torch.cuda.empty_cache()
    if stage=="ppo_training" and "optimizer_steps" not in result:
        progress_path=RESULT_DIR/"ppo_update_progress.json"
        result["optimizer_steps"] = json.loads(progress_path.read_text()).get("optimizer_steps",0) if progress_path.exists() else 0
    result.update({"stage": stage, "wall_time_seconds": time.perf_counter() - started,
                   "preflight": preflight, "peak_memory_allocated_mib": torch.cuda.max_memory_allocated() / 2**20,
                   "peak_memory_reserved_mib": torch.cuda.max_memory_reserved() / 2**20,
                   "peak_device_memory_mib": max(samples) if samples else None,
                   "device_sampling_interval_seconds":0.1, "chunk_size":TOKEN_CHUNK_SIZE, "micro_batch":1})
    result["device_peak_target_gib"] = 20.0
    result["device_peak_within_target"] = (result["peak_device_memory_mib"] / 1024 <= 20) if result["peak_device_memory_mib"] is not None else None
    result["functional_memory_target_pass"] = result.get("status") in ("completed","mechanism_passed") and result["device_peak_within_target"] is True
    _write_stage_json(report_path, result)
    return result


### 阶段交接：真实 Token、原始 Logprob 与 Adapter 指纹
Rollout 与 Training 使用独立进程和 Kernel；持久化完成后退出当前进程，再切换官方环境。


In [ ]:
def _file_sha(path: Path) -> str:
    import hashlib
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _sft_fingerprint(manifest: dict) -> dict:
    adapter = Path(resolve_manifest_path(manifest["adapter_path"]))
    return {"base_model_id": manifest["base_model"]["id"], "policy_version": manifest["policy_version"],
            "manifest_sha256": _file_sha(SFT_MANIFEST_PATH),
            "adapter_files": {p.name: _file_sha(p) for p in sorted(adapter.glob("adapter_*")) if p.is_file()}}


def _chosen_logprob_values(generated) -> list[float]:
    """vLLM 0.11 chosen-token dictionary lookup; never substitute another token."""
    raw, ids = generated.logprobs, list(generated.token_ids or [])
    if raw is None or not ids or len(raw) != len(ids):
        raise ValueError("Missing or misaligned vLLM token logprobs")
    result = []
    for token, candidates in zip(ids, raw):
        if not isinstance(candidates, dict) or token not in candidates:
            raise ValueError(f"Chosen token {token} missing from vLLM Logprob dictionary")
        value = float(candidates[token].logprob)
        if not math.isfinite(value): raise ValueError("non-finite token logprob")
        result.append(value)
    return result


def _persist_rollout(records: list[dict], path: Path, manifest_path: Path, sft: dict, expected_count: int) -> dict:
    if len(records) != expected_count: raise ValueError("Rollout count differs from Smoke budget")
    fingerprint = _sft_fingerprint(sft)
    _check_train_origin(records)
    for row in records:
        if row.get("fixture"): raise ValueError("CPU fixture cannot enter GPU pipeline")
        ids, completion = row["prompt_token_ids"], row["completion_token_ids"]
        if not ids or not completion or len(completion) != len(row["behavior_logprobs"]):
            raise ValueError("Invalid token alignment")
        if row["response_mask"] != [0] * len(ids) + [1] * len(completion):
            raise ValueError("Response Mask differs from stored Token IDs")
        row.update({"source_split": "train", "rollout_quantization": "bf16", "logprob_semantics": "vllm_v1_raw_logprobs",
                    "sampling_distribution": "raw_softmax_T1_topP1_topK_disabled", "sft_fingerprint": fingerprint})
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in records), encoding="utf-8")
    payload = {"schema_version": "nemotron-rollout/v2", "status": "completed", "records": len(records),
               "rollout_path": str(path.relative_to(PROJECT_ROOT)), "rollout_sha256": _file_sha(path),
               "sft_fingerprint": fingerprint, "source_split": "train", "temperature": 1.0, "top_p": 1.0,
               "top_k": -1, "logprobs_mode": "raw_logprobs", "quantization": "bf16",
               "max_new_tokens": MAX_NEW_TOKENS, "max_model_len": MAX_MODEL_LEN}
    _write_stage_json(manifest_path, payload)
    return payload


def _read_rollout(path: Path, manifest_path: Path) -> list[dict]:
    saved = json.loads(manifest_path.read_text())
    if saved.get("status") != "completed" or saved.get("rollout_sha256") != _file_sha(path):
        raise ValueError("Incomplete or modified Rollout artifact")
    if saved.get("sft_fingerprint") != _sft_fingerprint(read_sft_manifest()):
        raise ValueError("SFT Manifest/Adapter changed since Rollout; refuse policy-version mismatch")
    records = [json.loads(s) for s in path.read_text().splitlines() if s.strip()]
    if len(records) != saved["records"]: raise ValueError("Rollout record count mismatch")
    _check_train_origin(records)
    expected_fingerprint = saved["sft_fingerprint"]
    seen = set()
    for row in records:
        key = (row["sample_id"], row["group_index"])
        if key in seen: raise ValueError("Duplicate candidate index in Rollout group")
        seen.add(key)
        if row.get("sft_fingerprint") != expected_fingerprint or row.get("policy_version") != expected_fingerprint["policy_version"]:
            raise ValueError("Per-record SFT policy version mismatch")
        ids, completion = row["prompt_token_ids"], row["completion_token_ids"]
        if not ids or not completion or len(completion) != len(row["behavior_logprobs"]):
            raise ValueError("Invalid stored token/logprob alignment")
        if row["response_mask"] != [0]*len(ids)+[1]*len(completion):
            raise ValueError("Stored Response Mask mismatch")
        if not all(math.isfinite(float(x)) and float(x)<=1e-6 for x in row["behavior_logprobs"]):
            raise ValueError("Invalid sampled-token log probability")
        if row.get("source_split") != "train" or row.get("fixture"):
            raise ValueError("Only real Train Rollout may train")
        if row.get("temperature") != 1.0 or row.get("top_p") != 1.0:
            raise ValueError("This Smoke requires T=1, top_p=1 for raw/behavior logprob equivalence")
    return records


### 策略对齐：BF16 Rollout 与训练侧量化差异
Rollout 与 Training 使用独立进程和 Kernel；持久化完成后退出当前进程，再切换官方环境。


In [ ]:
def audit_policy_logprobs(model, tokenizer, records: list[dict], mode: str) -> dict:
    """Compare actual generated Token IDs without re-tokenizing the completion.

    One real sequence per task is teacher-forced. NF4 is explicitly a different
    target policy from the BF16 rollout; old denominator logprobs stay unchanged.
    """
    chosen, seen = [], set()
    for row in records:
        if row["task_type"] not in seen:
            chosen.append(row); seen.add(row["task_type"])
    deltas, checks = [], []
    model.eval()
    device = model.get_input_embeddings().weight.device
    for row in chosen:
        prompt_ids, response_ids = row["prompt_token_ids"], row["completion_token_ids"]
        rendered_ids = tokenizer.apply_chat_template([{"role":"user", "content":row["prompt"]}], tokenize=True, add_generation_prompt=True, enable_thinking=True)
        # Transformers 4 returns list[int]; Transformers 5 defaults to BatchEncoding.
        if hasattr(rendered_ids, "keys"):
            rendered_ids = rendered_ids["input_ids"]
        if list(rendered_ids) != prompt_ids: raise ValueError("Stored vLLM Prompt Token IDs differ from official Chat Template")
        if row["response_mask"] != [0] * len(prompt_ids) + [1] * len(response_ids):
            raise ValueError("Prompt/Response mask mismatch")
        ids = torch.tensor([prompt_ids + response_ids], device=device)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
            causal = model.get_base_model()
            hidden = causal.model(input_ids=ids, attention_mask=None, use_cache=False, return_dict=True).last_hidden_state
            values = []
            start = len(prompt_ids) - 1
            for offset in range(0, len(response_ids), TOKEN_CHUNK_SIZE):
                count = min(TOKEN_CHUNK_SIZE, len(response_ids) - offset)
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    block = model.get_output_embeddings()(hidden[:,start+offset:start+offset+count])
                labels = ids[:, start+offset+1:start+offset+count+1]
                selected = _selected_logprob(block, labels)
                values.extend(selected[0].cpu().tolist())
                del block, selected
        delta = torch.tensor(values) - torch.tensor(row["behavior_logprobs"], dtype=torch.float32)
        deltas.extend(delta.tolist())
        checks.append({"sample_id": row.get("sample_id", row.get("prompt_id")), "task_type": row["task_type"],
                       "response_tokens": len(response_ids), "prompt_tokens": len(prompt_ids),
                       "first_response_action_index": start, "token_mask_alignment": True,
                       "vllm_first_raw_logprob": row["behavior_logprobs"][0], "training_first_raw_logprob": values[0]})
        del hidden
    absolute = torch.tensor(deltas).abs()
    if not absolute.numel(): raise ValueError("No logprob evidence available")
    # Thresholds are explicit engineering gates, not equality claims or estimated KL.
    mean_limit, p99_limit = ((0.10, 0.60) if mode == "bf16_lora" else (0.50, 3.0))
    result = {"status": "passed" if absolute.mean() <= mean_limit and absolute.quantile(0.99) <= p99_limit else "blocked_policy_distribution_mismatch",
              "training_mode": mode, "rollout_mode": "bf16", "audit_autocast_dtype":"bfloat16", "scope": "one_actual_rollout_per_task",
              "tokens_compared": len(deltas), "mean_abs_logprob_delta": float(absolute.mean()),
              "p99_abs_logprob_delta": float(absolute.quantile(0.99)), "max_abs_logprob_delta": float(absolute.max()),
              "signed_mean_logprob_delta": float(torch.tensor(deltas).mean()), "mean_limit": mean_limit,
              "p99_limit": p99_limit, "quantization_difference_expected": mode == "nf4_qlora", "checks": checks}
    _write_stage_json(RESULT_DIR / f"alignment_{mode}.json", result)
    return result


### vLLM 0.11 原始概率与采样语义
显式关闭 top-k/top-p 裁剪、惩罚和模型 generation_config 默认值，使用 T=1。V1 raw_logprobs 才与行为采样分布相同；不从 top-1 代替缺失的采样 Token。


In [ ]:
def _generate_rollout_records(prompt_rows: list[dict], model_source: str, policy_version: str, adapter_path: str, n: int) -> list[dict]:
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest
    tokenizer = AutoTokenizer.from_pretrained(model_source, local_files_only=Path(model_source).exists())
    engine = None
    try:
        engine = LLM(model=model_source, tokenizer=model_source, dtype="bfloat16", enable_lora=True,
                     tensor_parallel_size=1, gpu_memory_utilization=float(os.environ.get("VLLM_GPU_MEMORY_UTILIZATION", "0.80")),
                     kv_cache_memory_bytes=VLLM_KV_CACHE_BYTES, max_num_batched_tokens=VLLM_MAX_BATCHED_TOKENS,
                     enable_chunked_prefill=True,
                     max_model_len=MAX_MODEL_LEN, max_num_seqs=VLLM_MAX_NUM_SEQS, enforce_eager=True, generation_config="vllm",
                     logprobs_mode="raw_logprobs", seed=SEED)
        rendered = [tokenizer.apply_chat_template([{"role":"user","content":r["prompt"]}], tokenize=False, add_generation_prompt=True, enable_thinking=True) for r in prompt_rows]
        prompt_lengths = [len(tokenizer.encode(s, add_special_tokens=False)) for s in rendered]
        if any(length + MAX_NEW_TOKENS > MAX_MODEL_LEN for length in prompt_lengths):
            raise ValueError("Context overflow: refuse silently reduced completion budget")
        budgets = [MAX_NEW_TOKENS for _ in prompt_lengths]
        if any(budget < 1 for budget in budgets): raise ValueError("Prompt alone exhausts max_model_len")
        sampling = [SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=-1, repetition_penalty=1.0,
                                  presence_penalty=0.0, frequency_penalty=0.0, n=n, max_tokens=budget,
                                  logprobs=1, seed=SEED, skip_special_tokens=False) for budget in budgets]
        outputs = engine.generate(rendered, sampling, lora_request=LoRARequest("sft", 1, adapter_path), use_tqdm=True)
        if len(outputs) != len(prompt_rows): raise ValueError("Rollout request count mismatch")
        rows = []
        for original, request, budget in zip(prompt_rows, outputs, budgets):
            if len(request.outputs) != n: raise ValueError("Rollout group size mismatch")
            prompt_ids = list(request.prompt_token_ids or [])
            for index, out in enumerate(request.outputs):
                ids = list(out.token_ids or [])
                rows.append({"sample_id":str(original["id"]), "prompt_id":str(original["id"]),
                    "task_type":original["task_type"], "prompt":original["prompt"], "expected_answer":str(original["answer"]),
                    "completion":out.text, "prompt_token_ids":prompt_ids, "completion_token_ids":ids,
                    "response_mask":[0]*len(prompt_ids)+[1]*len(ids), "behavior_logprobs":_chosen_logprob_values(out),
                    "finish_reason":out.finish_reason, "stop_reason":out.stop_reason, "truncated":out.finish_reason=="length",
                    "policy_version":policy_version, "temperature":1.0, "top_p":1.0, "top_k":-1,
                    "effective_max_new_tokens":budget, "max_model_len":MAX_MODEL_LEN, "max_num_seqs":VLLM_MAX_NUM_SEQS, "kv_cache_budget_bytes":VLLM_KV_CACHE_BYTES,
                    "group_index":index, "source":"vllm_rollout", "fixture":False, "trace_verified":False})
        return rows
    finally:
        if engine is not None: del engine
        import gc
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


### Adapter 磁盘重载检查
更新后先保存，再从磁盘加载为独立 PEFT Adapter，比较真实模型 logits；不把目录存在等同于重载成功。


In [ ]:
def actual_adapter_reload(model, adapter_dir: Path, input_ids: list[int]) -> dict:
    """Load saved PEFT weights from disk under a second adapter and compare logits."""
    model.eval()
    original = model.active_adapter
    if isinstance(original, list): original = original[0]
    device = model.get_input_embeddings().weight.device
    probe = torch.tensor([input_ids[:32]], device=device)
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
        before = model(input_ids=probe, attention_mask=torch.ones_like(probe), use_cache=False).logits.float().cpu()
    model.load_adapter(str(adapter_dir), adapter_name="disk_reload_check", is_trainable=False)
    model.set_adapter("disk_reload_check")
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
        after = model(input_ids=probe, attention_mask=torch.ones_like(probe), use_cache=False).logits.float().cpu()
    maximum = float((before - after).abs().max())
    model.set_adapter(original)
    model.delete_adapter("disk_reload_check")
    if maximum > 1e-5: raise ValueError(f"Adapter reload logit mismatch: {maximum}")
    return {"status":"passed", "method":"PEFT load_adapter from saved disk weights, compare real-model logits", "max_abs_logit_difference":maximum}


### CPU 机制测试：BF16 Hidden State 与 FP32 Value Head
此处只检查 dtype/device 与反向传播，不代表真实模型训练。


In [ ]:
class BF16BackboneFixture(nn.Module):
    def __init__(self):
        super().__init__()
        self.config = type("Config", (), {"hidden_size":8})()
        self.embedding = nn.Embedding(16,8,dtype=torch.bfloat16)
        self.head = nn.Linear(8,16,dtype=torch.bfloat16)
    def forward(self,input_ids,attention_mask=None,output_hidden_states=True,use_cache=False):
        hidden=self.embedding(input_ids)
        return type("Output",(),{"logits":self.head(hidden),"hidden_states":[hidden]})()

head_fixture = CausalLMValueHead(BF16BackboneFixture())
fixture_ids = torch.tensor([[1,2,3]])
fixture_output = head_fixture(fixture_ids)
assert fixture_output["values"].dtype == torch.float32
assert fixture_output["values"].device == fixture_output["logits"].device
fixture_output["values"].square().mean().backward()
assert head_fixture.value_head.weight.grad is not None
_write_stage_json(RESULT_DIR / "cpu_value_head_dtype_test.json", {"status":"passed","fixture_only":True,"input_dtype":"bfloat16","value_dtype":"float32","backward":True})


### 来源检查：Train 原题和候选索引
持久化与重新读取时核对原始 Train ID、Prompt、答案和任务；不读取 Holdout。


In [ ]:
def _check_train_origin(records):
    import pandas as pd
    frame = pd.read_csv(TRAIN_PATH,dtype=str,keep_default_na=False)
    if set(frame["split"]) != {"train"}: raise ValueError("Train split metadata mismatch")
    lookup = frame.set_index("id").to_dict("index")
    for row in records:
        original = lookup.get(str(row["sample_id"]))
        if original is None or row["prompt"] != original["prompt"] or str(row["expected_answer"]) != original["answer"] or row["task_type"] != original["task_type"]:
            raise ValueError("Rollout is not an unchanged official Train instance")


## GPU Smoke 独立执行入口：PPO

只开启一个开关：`RUN_PPO_MEMORY_TEST=1`（真实模型 16K 机制测试，无 Optimizer Step）、`RUN_PPO_ROLLOUT=1`（BF16 vLLM），或 `RUN_PPO_TRAINING=1`（NF4 QLoRA）。切换 Profile 前结束上一个进程并确认显存释放。`PPO_SFT_MANIFEST` 指向共同父 SFT，`PPO_RUN_ID` 指定本轮独立结果/Checkpoint 目录；默认均不运行 GPU。执行说明和实测报告见 `docs/06_PPO.md`。


### CPU 检查：真实 Train 采样入口
在启动模型前运行同一个采样函数，避免仅测试 Fixture 而漏掉导入或数据路径错误。

In [ ]:
_selected_ppo_train = select_ppo_smoke_prompts()
assert Counter(r["task_type"] for r in _selected_ppo_train) == {"equation_numeric": 2, "bit_manipulation": 2}
_check_train_origin([dict(r, sample_id=r["id"], expected_answer=r["answer"]) for r in _selected_ppo_train])
print({"train_selection": "passed", "records": len(_selected_ppo_train)})


### 数值回归与 16K 显存验收

下面的小 Tensor 测试比较旧实现与分块实现的数值及梯度，并检查完整 PPO Loss、Padding 和累积归约。GPU 显存机制入口默认关闭：它使用真实 NF4 模型和人工拼接的 16K 有效输入，只做前后向及累积，不执行 Optimizer Update。真实更新由独立 Training 入口在 Rollout、Verifier 和 Logprob 审计通过后执行。

In [ ]:
def run_chunk_equivalence_tests():
    """Dense pre-optimization references are confined to small CPU tensors."""
    torch.manual_seed(SEED)
    checks, errors = [], {}
    def close(name, actual, expected):
        torch.testing.assert_close(actual, expected, rtol=3e-5, atol=3e-6)
        errors[name] = float((actual.detach().float()-expected.detach().float()).abs().max())
        checks.append(name)
    for chunk in (1, 3, 128):
        labels = torch.randint(0, 19, (2, 9))
        mask = torch.tensor([[0,0,1,1,1,1,0,0],[0,1,1,1,1,1,1,1]], dtype=torch.float32)
        dense = torch.randn(2,9,19, requires_grad=True)
        blocks = dense.detach().clone().requires_grad_()
        lp_ref = F.log_softmax(dense[:,:-1].float(),-1).gather(-1,labels[:,1:,None]).squeeze(-1)
        lp_new = token_logprobs(blocks,labels,chunk)
        close(f"logprob_{chunk}",lp_new,lp_ref)
        grad_ref = torch.autograd.grad((lp_ref*mask).sum(),dense)[0]
        grad_new = torch.autograd.grad((lp_new*mask).sum(),blocks)[0]
        close(f"logprob_gradient_{chunk}",grad_new,grad_ref)
        assert torch.count_nonzero(grad_new[:,:-1][~mask.bool()]) == 0
        assert torch.count_nonzero(grad_new[:,-1]) == 0
        dense = torch.randn(2,8,19, requires_grad=True)
        blocks = dense.detach().clone().requires_grad_()
        h_ref = masked_mean(-(F.softmax(dense.float(),-1)*F.log_softmax(dense.float(),-1)).sum(-1),mask)
        h_new = entropy_from_logits(blocks,mask,chunk)
        close(f"entropy_{chunk}",h_new,h_ref)
        close(f"entropy_gradient_{chunk}",torch.autograd.grad(h_new,blocks)[0],torch.autograd.grad(h_ref,dense)[0])
        # Same complete PPO objective, including value and old/new KL diagnostic.
        dense = torch.randn(2,9,19, requires_grad=True)
        blocks = dense.detach().clone().requires_grad_()
        vr = torch.randn(2,8,requires_grad=True); vn=vr.detach().clone().requires_grad_()
        old_lp = torch.randn(2,8)-3
        rewards = torch.zeros(2,8); rewards[0,5]=1; rewards[1,7]=1
        done = torch.zeros_like(mask); done[0,5]=1;done[1,7]=1
        advantages, returns = compute_gae(rewards,torch.randn(2,8),done,mask)
        advantages = normalize_masked(advantages,mask)
        def total(z,v,optimized):
            lp = token_logprobs(z,labels,chunk) if optimized else F.log_softmax(z[:,:-1].float(),-1).gather(-1,labels[:,1:,None]).squeeze(-1)
            entropy = entropy_from_logits(z[:,:-1],mask,chunk) if optimized else masked_mean(-(F.softmax(z[:,:-1].float(),-1)*F.log_softmax(z[:,:-1].float(),-1)).sum(-1),mask)
            return ppo_clip_loss(old_lp,lp,advantages,mask)["policy_loss"]+VALUE_COEF*value_loss(v,returns,mask)-ENTROPY_COEF*entropy+KL_COEF*approx_kl(old_lp,lp,mask)
        reference, optimized=total(dense,vr,False),total(blocks,vn,True)
        close(f"ppo_total_loss_{chunk}",optimized,reference)
        gr=torch.autograd.grad(reference,(dense,vr));gn=torch.autograd.grad(optimized,(blocks,vn))
        for name,a,b in zip(("logits","value"),gn,gr):close(f"ppo_gradient_{name}_{chunk}",a,b)
    # Projection checkpoint is also checked against a dense LM Head, including gradients.
    hr=torch.randn(2,8,7,requires_grad=True);hn=hr.detach().clone().requires_grad_()
    head=nn.Linear(7,19,bias=False)
    ref_logits=head(hr)
    ref_lp=F.log_softmax(ref_logits,-1).gather(-1,labels[:,1:,None]).squeeze(-1)
    ref_h=masked_mean(-(F.softmax(ref_logits,-1)*F.log_softmax(ref_logits,-1)).sum(-1),mask)
    new_lp,new_h=projected_policy_statistics(hn,head,labels[:,1:],mask,3)
    close("projected_logprob",new_lp,ref_lp);close("projected_entropy",new_h,ref_h)
    gr=torch.autograd.grad((ref_lp*mask).sum()-ref_h,(hr,head.weight))
    gn=torch.autograd.grad((new_lp*mask).sum()-new_h,(hn,head.weight))
    for name,a,b in zip(("hidden","head"),gn,gr):close(f"projected_gradient_{name}",a,b)
    # Token-weighted accumulation must equal the unsplit batch objective.
    import copy
    rollout=build_cpu_rollout();rollout.response_mask[0,-2:]=0;rollout.dones[0,-1]=0;rollout.rewards[0,-1]=0
    whole=TinyActorCritic();micro=copy.deepcopy(whole)
    r,v,d,m=align_ppo_action_axis(rollout.rewards,rollout.old_values,rollout.dones,rollout.response_mask)
    advantage,_=compute_gae(r,v,d,m); active=advantage[m.bool()]; stats=(active.mean(),active.var(unbiased=False))
    ow=torch.optim.SGD(whole.parameters(),lr=1);om=torch.optim.SGD(micro.parameters(),lr=1)
    ppo_update(whole,ow,rollout,step_optimizer=False,advantage_stats=stats)
    total_tokens=m.sum()
    for index in range(2):
        fields={name:(getattr(rollout,name)[index:index+1] if isinstance(getattr(rollout,name),torch.Tensor) else getattr(rollout,name)) for name in rollout.__dataclass_fields__}
        item=PPORollout(**fields)
        ppo_update(micro,om,item,step_optimizer=False,loss_scale=float(m[index].sum()/total_tokens),advantage_stats=stats)
    for (name,a),(_,b) in zip(whole.named_parameters(),micro.named_parameters()):close(f"accumulation_{name}",a.grad,b.grad)
    # Empty response masks are safe and contribute exactly zero gradient.
    z=torch.randn(1,5,19,requires_grad=True);zero=entropy_from_logits(z,torch.zeros(1,5),3)
    close("zero_mask_entropy",zero,torch.tensor(0.));assert torch.count_nonzero(torch.autograd.grad(zero,z)[0])==0
    require_nf4_manifest({"quantization":{"mode":"nf4_qlora","load_in_4bit":True,"quant_type":"nf4","double_quant":True,"compute_dtype":"bfloat16"}})
    try: require_nf4_manifest({"quantization":{"mode":"bf16_lora"}})
    except ValueError: checks.append("reject_bf16_manifest")
    else: raise AssertionError("BF16 Manifest incorrectly accepted")
    from types import SimpleNamespace
    quant={"bnb_4bit_quant_type":"nf4","bnb_4bit_use_double_quant":True,"bnb_4bit_compute_dtype":"bfloat16"}
    base=SimpleNamespace(config=SimpleNamespace(quantization_config=quant,use_cache=False),is_loaded_in_4bit=True,is_gradient_checkpointing=True)
    wrapper=SimpleNamespace(base_model=SimpleNamespace(get_base_model=lambda:base),value_head=nn.Linear(3,1,dtype=torch.float32))
    assert all(assert_nf4_training_model(wrapper).values());checks.append("runtime_nf4_contract")
    base.is_loaded_in_4bit=False
    try: assert_nf4_training_model(wrapper)
    except ValueError: checks.append("runtime_reject_bf16_policy")
    else: raise AssertionError("Unquantized training policy incorrectly accepted")
    report={"status":"passed","scope":"CPU numerical equivalence, not GPU evidence","checks":checks,"max_absolute_errors":errors,"chunk_sizes":[1,3,128],"rtol":3e-5,"atol":3e-6}
    _write_stage_json(RESULT_DIR/"chunk_numerical_equivalence.json",report)
    return report

chunk_equivalence_report=run_chunk_equivalence_tests()
print({"chunk_equivalence":chunk_equivalence_report["status"],"checks":len(chunk_equivalence_report["checks"])})

In [ ]:
def run_ppo_memory_mechanism():
    """Selected real model, artificial 16K inputs. Never an outcome/learning experiment."""
    if not RUN_PPO_MEMORY_TEST: return {"status":"skipped_by_default"}
    def operation():
        from transformers import AutoTokenizer
        sft=read_sft_manifest()
        model=_load_sft_actor_critic(sft,"nf4_qlora")
        tokenizer=AutoTokenizer.from_pretrained(os.environ.get("QWEN3_MODEL_PATH",sft.get("tokenizer_path",sft["base_model"]["path"])))
        # Repeat real Train token content only to exercise 16384 ACTIVE positions.
        # This is deliberately not represented as a generated/verified Rollout.
        prompt=select_ppo_smoke_prompts()[0]["prompt"]
        seed_ids=tokenizer.encode(prompt,add_special_tokens=False)
        length=MEMORY_TEST_SEQUENCE_LENGTH
        ids=torch.tensor([(seed_ids*((length+len(seed_ids)-1)//len(seed_ids)))[:length]],device=model.value_head.weight.device)
        attention=torch.ones_like(ids); mask=torch.ones_like(ids);mask[:,:256]=0
        model.eval()
        with torch.no_grad(),torch.autocast("cuda",dtype=torch.bfloat16):
            old=model(ids,attention,response_mask=mask)
            old_lp=torch.zeros_like(ids,dtype=torch.float32);old_lp[:,1:]=old["logprobs"]
            old_values=old["values"].detach()
        del old
        reward=torch.zeros_like(old_lp);done=torch.zeros_like(old_lp);done[:,-1]=1
        fixture=PPORollout(ids,attention,mask,old_lp,old_values,reward,done,"memory_mechanism_fixture","not_applicable","stop",False,"mechanism_only","not_a_rollout")
        optimizer=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),lr=LEARNING_RATE)
        model.zero_grad(set_to_none=True)
        r,v,d,m=align_ppo_action_axis(reward,old_values,done,mask)
        a,_=compute_gae(r,v,d,m);active=a[m.bool()];stats=(active.mean(),active.var(unbiased=False))
        logs=[]
        for _ in range(2):
            logs.append(ppo_update(model,optimizer,fixture,step_optimizer=False,loss_scale=0.5,advantage_stats=stats))
        grads={"value_head":bool(model.value_head.weight.grad is not None and torch.isfinite(model.value_head.weight.grad).all()),
               "lora":any(p.grad is not None and bool(torch.isfinite(p.grad).all()) for n,p in model.named_parameters() if "lora_" in n)}
        if not all(grads.values()): raise ValueError("Missing/non-finite real-model gradients")
        grad_norms={"value_head":float(model.value_head.weight.grad.float().norm()),
                    "lora":float(torch.sqrt(sum(p.grad.float().square().sum() for n,p in model.named_parameters() if "lora_" in n and p.grad is not None)))}
        _write_stage_json(RESULT_DIR/"memory_core_forward_backward.json", {"status":"passed", "sequence_length":length, "active_attention_tokens":int(attention.sum()), "gradient_accumulation_micro_steps":2, "optimizer_steps":0, "gradients":grads, "gradient_norms":grad_norms, "logs":logs, "fixture":True})
        # No optimizer.step: without a verified real Rollout, weights must not update.
        snapshot=CHECKPOINT_ROOT/"ppo_memory_unchanged_snapshot"
        snapshot.mkdir(parents=True,exist_ok=True)
        model.base_model.save_pretrained(snapshot)
        torch.save(model.value_head.state_dict(),snapshot/"value_head.pt")
        reload_check=actual_adapter_reload(model.base_model,snapshot,ids[0].tolist())
        head=nn.Linear(model.value_head.in_features,1,bias=False).to(model.value_head.weight.device,dtype=torch.float32)
        head.load_state_dict(torch.load(snapshot/"value_head.pt",weights_only=True,map_location=model.value_head.weight.device))
        torch.testing.assert_close(head.weight,model.value_head.weight,rtol=0,atol=0)
        return {"status":"mechanism_passed","scope":"real NF4 model + artificial full-length input; NOT a real PPO Update", "fixture":True,
                "sequence_length":length,"active_attention_tokens":int(attention.sum()),"response_tokens":int(m.sum()),"micro_batch":1,
                "gradient_accumulation_micro_steps":2,"optimizer_steps":0,"ppo_update_status":"Not Run: requires verified real Rollout",
                "chunk_size":TOKEN_CHUNK_SIZE,"training_mode":"nf4_qlora","nf4_runtime_contract":model.nf4_contract,"value_head_dtype":str(model.value_head.weight.dtype),
                "hidden_dtype":model.last_hidden_dtype,"gradients":grads,"gradient_norms":grad_norms,"logs":logs,"adapter_reload":reload_check,
                "value_head_reload":"passed","snapshot_scope":"unchanged initialized weights, not a trained checkpoint"}
    return _gpu_stage("ppo_memory_mechanism",operation)

In [ ]:
if sum((RUN_PPO_ROLLOUT, RUN_PPO_TRAINING, RUN_PPO_MEMORY_TEST)) > 1:
    ppo_gpu_status = {"status":"blocked_kernel_switch_required", "reason":"分别在 vLLM 和训练环境执行；两阶段开关不能同时开启"}
elif RUN_PPO_MEMORY_TEST:
    ppo_gpu_status = run_ppo_memory_mechanism()
elif RUN_PPO_ROLLOUT:
    ppo_gpu_status = run_ppo_rollout_phase()
elif RUN_PPO_TRAINING:
    ppo_gpu_status = run_ppo_training_phase(os.environ.get("RL_QUANTIZATION_MODE", "nf4_qlora"))
else:
    ppo_gpu_status = {"status":"skipped_by_default"}
print(json.dumps(ppo_gpu_status,ensure_ascii=False,indent=2))


### 课程实验：失败样本和长答也进入 PPO
沿用上面的 PPO Clip、Value Loss 和 GAE。输出达到预算视为有限时域任务结束，末端价值置零；不伪装为观察到 EOS。gamma=1、lambda=1 将可验证结果传到全段动作。


In [ ]:
def course_ppo_outcome(rollout, record, length_weight=0.15):
    # Every sampled outcome is eligible, including incorrect/truncated answers.
    length = len(record["completion_token_ids"])
    cap = int(record["max_tokens"])
    correct = bool(record["correct"]) and not record["truncated"]
    reward = (1.0 - length_weight * min(length / cap, 1.0)) if correct else 0.0
    rollout.rewards.zero_()
    positions = torch.nonzero(rollout.response_mask[0], as_tuple=False).flatten()
    rollout.rewards[0, int(positions[-1])] = reward
    # GAE masks the unobserved next state. Never overwrite an observed state value.
    return rollout


## 第二轮：Equation PPO Reward 与正式配置
仅 Equation Numeric 训练；Roman、Gravity 只做配对能力保持评测。PPO 与 GRPO 从同一个 SFT 独立初始化。
本轮 γ=λ=1，有限输出预算作为任务终点；实际 EOS 单独记录。GAE 自动将不可见末态的 bootstrap 置零，**不清零最后一个已观察状态的 V**。Value 是预测下一 action 前的 hidden state。
唯一复合 Reward R1=F+.20P+.05S 的数学定义在 Notebook 06 的 `equation_reward_components`；此处通过维护者 loader 读取同一函数。Verifier 在 Notebook 01 的 `audit_equation_process`。模型自评不参与 Reward；Format/Process 由程序验证后按固定小权重加入奖励；Length只统计，不做奖励消融。
PPO 不按 GRPO 的组内方差筛选数据。全错 batch 仍可能有 Critic 校正和相对 Value 的 advantage；不得预设为零梯度。Reference KL 尚未实现，系数只允许 0；Old Policy Approx KL 仅诊断。
当前 Runner 使用显式 BF16 LoRA；前文 NF4 standalone Smoke 是历史路径，不是第二轮正式入口。

In [ ]:
# Shared teaching definition stays in Notebook 06; .py only extracts notebook APIs.
from utils.course_worker import notebook_api as _course_notebook_api
equation_reward_components = _course_notebook_api("grpo")["equation_reward_components"]